# Marginal MCMC — PROMIS Neuropathic Pain (Factorized GRM)

Runs NUTS on item parameters with abilities Rao-Blackwellized out,
starting from the fitted ADVI baseline model.
Data from Harvard Dataverse (doi:10.7910/DVN/TJ9MNM).

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['JAX_ENABLE_X64'] = '1'
os.environ['TQDM_DISABLE'] = '1'
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import jax
import jax.numpy as jnp

## 1. Load Data and Fitted ADVI Model

In [ ]:
from bayesianquilts.data.promis_neuropathic_pain import get_multidomain_data, item_keys, response_cardinality

df, num_people, scale_indices = get_multidomain_data(polars_out=True, min_items=10)
print(f"Dataset: {num_people} people, {len(item_keys)} items, {len(scale_indices)} domains")
for domain, indices in scale_indices.items():
    print(f"  {domain}: {len(indices)} items")

def make_data_dict(dataframe):
    data = {}
    for col in dataframe.columns:
        data[col] = dataframe[col].to_numpy().astype(np.float64)
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

data = make_data_dict(df)

In [ ]:
from bayesianquilts.irt.factorizedgrm import FactorizedGRModel

model = FactorizedGRModel.load_from_disk('grm_baseline')
print(f"Loaded ADVI baseline model: {len(model.item_keys)} items, {model.dimensions} dimensions")

## 2. Attach Imputation Model

In [ ]:
from pathlib import Path

pairwise_path = Path('pairwise_stacking_model.yaml')
if pairwise_path.exists():
    from bayesianquilts.imputation.pairwise_stacking import PairwiseOrdinalStackingModel
    pairwise_model = PairwiseOrdinalStackingModel.load(str(pairwise_path))
    model.imputation_model = pairwise_model

    pmfs, weights = model._compute_batch_pmfs(data)
    if pmfs is not None:
        data['_imputation_pmfs'] = pmfs
        if weights is not None:
            data['_imputation_weights'] = weights
    print("Imputation model attached")
else:
    print("No imputation model found — running without imputation")

## 3. Run Marginal MCMC (NUTS)

In [ ]:
NUM_CHAINS = 4
NUM_WARMUP = 500
NUM_SAMPLES = 500

mcmc_samples = model.fit_marginal_mcmc(
    data,
    theta_grid=None,
    num_chains=NUM_CHAINS,
    num_warmup=NUM_WARMUP,
    num_samples=NUM_SAMPLES,
    target_accept_prob=0.85,
    step_size=0.01,
    seed=42,
    verbose=True,
)

for var_name, samples in mcmc_samples.items():
    flat = np.array(samples).reshape(-1, *samples.shape[2:])
    print(f"{var_name}: shape={samples.shape}, mean={np.mean(flat):.4f}, std={np.std(flat):.4f}")

## 4. Compute EAP Abilities

In [ ]:
eap_result = model.compute_eap_abilities(data)
eap = np.array(eap_result['eap'])
psd = np.array(eap_result['psd'])
print(f"EAP abilities: mean={np.mean(eap):.4f}, std={np.std(eap):.4f}")
print(f"Mean PSD: {np.mean(psd):.4f}")

## 5. Standardize and Fit Surrogate

In [ ]:
stats = model.standardize_marginal(data)
print(f"Standardization: mu={stats['mu']:.4f}, sigma={stats['sigma']:.4f}")

eap_result = model.compute_eap_abilities(data)
eap_std = np.array(eap_result['eap'])
print(f"Post-standardization EAP: mean={np.mean(eap_std):.4f}, std={np.std(eap_std):.4f}")

model.fit_surrogate_to_mcmc()

# Inject EAP abilities into surrogate_sample
eap_arr = np.array(eap_result['eap'])
model.surrogate_sample['abilities'] = jnp.array(
    eap_arr[:, np.newaxis, np.newaxis, np.newaxis]
)[np.newaxis, ...]

## 6. MCMC Diagnostics

In [ ]:
import pandas as pd

diag_rows = []
for var_name, samples in mcmc_samples.items():
    flat = np.array(samples).reshape(-1, *samples.shape[2:])
    row = {
        'Parameter': var_name,
        'Mean': f"{np.mean(flat):.4f}",
        'Std': f"{np.std(flat):.4f}",
        'Min': f"{np.min(flat):.4f}",
        'Max': f"{np.max(flat):.4f}",
    }
    if samples.shape[0] > 1:
        chain_means = np.mean(np.array(samples), axis=1)
        between_var = np.var(chain_means, axis=0, ddof=1)
        within_var = np.mean(np.var(np.array(samples), axis=1, ddof=1), axis=0)
        n = samples.shape[1]
        r_hat = np.sqrt(((n - 1) / n * within_var + between_var) / np.maximum(within_var, 1e-30))
        row['R-hat (mean)'] = f"{np.mean(r_hat):.4f}"
        row['R-hat (max)'] = f"{np.max(r_hat):.4f}"
    diag_rows.append(row)
print(pd.DataFrame(diag_rows).to_string(index=False))

In [ ]:
# Compare ADVI vs MCMC posterior means
if model.params is not None:
    print("ADVI vs MCMC comparison:")
    for var_name in mcmc_samples:
        mcmc_mean = np.mean(
            np.array(mcmc_samples[var_name]).reshape(-1, *mcmc_samples[var_name].shape[2:]),
            axis=0
        )
        loc_key = None
        for pk in model.params:
            if pk.startswith(var_name) and pk.endswith('loc'):
                loc_key = pk
                break
        if loc_key is not None:
            advi_mean = np.array(model.params[loc_key])
            diff = np.abs(mcmc_mean.squeeze() - advi_mean.squeeze())
            print(f"  {var_name}: |MCMC - ADVI| mean={np.mean(diff):.4f}, max={np.max(diff):.4f}")

## 7. Save

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(eap_std, bins=50, density=True, alpha=0.7)
axes[0].set_title('EAP Ability Distribution (post-standardization)')
axes[0].set_xlabel('θ')

axes[1].scatter(eap_std, np.array(eap_result['psd']), alpha=0.3, s=5)
axes[1].set_title('EAP vs PSD')
axes[1].set_xlabel('θ (EAP)')
axes[1].set_ylabel('PSD')

fig.suptitle(f'PROMIS Neuropathic Pain — MCMC EAP Abilities')
fig.tight_layout()
fig.savefig('mcmc_eap_abilities.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
model.save_to_disk('grm_mcmc')

# Save standalone NPZ
os.makedirs('mcmc_samples', exist_ok=True)
save_dict = {}
for var_name, samples in mcmc_samples.items():
    save_dict[var_name] = np.array(samples)
save_dict['eap'] = eap
save_dict['psd'] = np.array(eap_result['psd'])
save_dict['eap_standardized'] = eap_std
save_dict['psd_standardized'] = np.array(eap_result['psd'])
save_dict['standardize_mu'] = stats['mu']
save_dict['standardize_sigma'] = stats['sigma']
np.savez('mcmc_samples/mcmc_item_params.npz', **save_dict)
print("Saved MCMC model and samples")

## Summary\n\nRan marginal MCMC (NUTS with Rao-Blackwellized abilities) on the\nPROMIS Neuropathic Pain factorized GRM item parameters, starting from ADVI\nposterior means. Item parameters sampled with 4 chains × 500 samples.\nEAP abilities computed via numerical integration on the MCMC posterior.